# DSPy Modules Deep Dive

Modules are the building blocks for composing language model programs in DSPy.

**What You'll Learn:**
- How `dspy.Module` works (`__init__` and `forward`)
- Wrapping predictors in modules
- Composing modules (module calling module)
- Inspecting module parameters with `named_predictors()`
- Modules with conditional logic

## Setup

In [ ]:
import os
import sys
sys.path.append('../../')

import dspy
from utils import setup_default_lm, print_step, print_result, print_error, configure_dspy
from dotenv import load_dotenv

load_dotenv('../../.env')

try:
    lm = setup_default_lm(provider="openai", model="gpt-4o", max_tokens=500)
    configure_dspy(lm=lm)
    print_result("Language model configured successfully!", "Status")
except Exception as e:
    print_error(f"Failed to configure language model: {e}")
    print("Make sure you have set your OPENAI_API_KEY in the .env file")

## Example 1: Simplest Module

The minimal DSPy module wraps a single `Predict` call.

In [ ]:
class SimpleQA(dspy.Module):
    """The simplest possible DSPy module."""
    def __init__(self):
        super().__init__()

        class QASignature(dspy.Signature):
            """Answer the question concisely."""
            question = dspy.InputField()
            answer = dspy.OutputField()

        self.predict = dspy.Predict(QASignature)

    def forward(self, question):
        return self.predict(question=question)

qa = SimpleQA()
result = qa(question="What is the speed of light?")
print_result(f"Answer: {result.answer}", "Simple Module")

## Example 2: Multiple Predictors

A module that chains two LM calls: translate, then summarize.

In [ ]:
class TranslateAndSummarize(dspy.Module):
    """Translate text to English, then summarize it."""
    def __init__(self):
        super().__init__()

        class Translate(dspy.Signature):
            """Translate the given text to English."""
            text = dspy.InputField(desc="Text in any language")
            english_text = dspy.OutputField(desc="English translation")

        class Summarize(dspy.Signature):
            """Summarize the text in one sentence."""
            text = dspy.InputField()
            summary = dspy.OutputField(desc="One-sentence summary")

        self.translator = dspy.Predict(Translate)
        self.summarizer = dspy.Predict(Summarize)

    def forward(self, text):
        translated = self.translator(text=text)
        summarized = self.summarizer(text=translated.english_text)
        return dspy.Prediction(
            english_text=translated.english_text,
            summary=summarized.summary
        )

pipeline = TranslateAndSummarize()
result = pipeline(text="La inteligencia artificial está transformando la industria tecnológica a nivel mundial.")

print_result(
    f"Translation: {result.english_text}\nSummary: {result.summary}",
    "Translate + Summarize"
)

## Example 3: Module Composition

Modules can use other modules — just instantiate them in `__init__` and call them in `forward`.

In [ ]:
class Classifier(dspy.Module):
    """Classify text into a category."""
    def __init__(self):
        super().__init__()

        class ClassifyText(dspy.Signature):
            """Classify the text into one of: technology, science, sports, politics, entertainment."""
            text = dspy.InputField()
            category = dspy.OutputField(desc="One of: technology, science, sports, politics, entertainment")

        self.classify = dspy.Predict(ClassifyText)

    def forward(self, text):
        return self.classify(text=text)

class CategoryExpert(dspy.Module):
    """Generate expert analysis based on text and its category."""
    def __init__(self):
        super().__init__()
        self.classifier = Classifier()  # Reuse another module

        class ExpertAnalysis(dspy.Signature):
            """Provide expert analysis of the text from the perspective of the given category."""
            text = dspy.InputField()
            category = dspy.InputField()
            analysis = dspy.OutputField(desc="Expert analysis from the category perspective")

        self.analyzer = dspy.ChainOfThought(ExpertAnalysis)

    def forward(self, text):
        classification = self.classifier(text=text)
        analysis = self.analyzer(text=text, category=classification.category)
        return dspy.Prediction(
            category=classification.category,
            analysis=analysis.analysis,
            reasoning=analysis.reasoning
        )

expert = CategoryExpert()
result = expert(text="SpaceX successfully launched its Starship rocket into orbit for the first time.")

print_result(
    f"Category: {result.category}\nAnalysis: {result.analysis}",
    "Composed Modules"
)

## Example 4: Inspecting Module Parameters

Use `named_predictors()` to see the structure of a module.

In [ ]:
print("CategoryExpert module predictors:")
for name, predictor in expert.named_predictors():
    print(f"  - {name}: {type(predictor).__name__}")

print(f"\nTranslateAndSummarize module predictors:")
for name, predictor in pipeline.named_predictors():
    print(f"  - {name}: {type(predictor).__name__}")

## Example 5: Conditional Logic in Modules

The `forward()` method can contain any Python logic — including conditionals for adaptive behavior.

In [ ]:
class AdaptiveQA(dspy.Module):
    """Answer questions using simple or detailed mode based on complexity."""
    def __init__(self):
        super().__init__()

        class AssessComplexity(dspy.Signature):
            """Assess whether the question is simple or complex."""
            question = dspy.InputField()
            complexity = dspy.OutputField(desc="Either 'simple' or 'complex'")

        class SimpleAnswer(dspy.Signature):
            """Give a brief, direct answer."""
            question = dspy.InputField()
            answer = dspy.OutputField(desc="A brief one-line answer")

        class DetailedAnswer(dspy.Signature):
            """Give a thorough answer with explanation."""
            question = dspy.InputField()
            answer = dspy.OutputField(desc="A detailed answer with explanation")

        self.assess = dspy.Predict(AssessComplexity)
        self.simple_qa = dspy.Predict(SimpleAnswer)
        self.detailed_qa = dspy.ChainOfThought(DetailedAnswer)

    def forward(self, question):
        assessment = self.assess(question=question)

        if "complex" in assessment.complexity.lower():
            result = self.detailed_qa(question=question)
            return dspy.Prediction(answer=result.answer, mode="detailed", reasoning=result.reasoning)
        else:
            result = self.simple_qa(question=question)
            return dspy.Prediction(answer=result.answer, mode="simple", reasoning="N/A")

adaptive = AdaptiveQA()

for question in ["What color is the sky?", "How does photosynthesis convert light energy into chemical energy at the molecular level?"]:
    result = adaptive(question=question)
    print_result(f"Q: {question}\nMode: {result.mode}\nAnswer: {result.answer}", "Adaptive QA")

## Summary

**Key Takeaways:**
1. Modules inherit from `dspy.Module` and implement `forward()`
2. Predictors (`Predict`, `ChainOfThought`) are set up in `__init__()`
3. Modules can compose other modules for complex pipelines
4. Use `named_predictors()` to inspect module structure
5. `forward()` can contain conditional logic for adaptive behavior